# Regularization and Training Stabilizers: Dropout, DropPath, Soft-Capping, and Router Z-Loss (C03 Topic 06)

**Citations:** Srivastava et al. (2014 Dropout); Huang et al. (2016 DropPath); Szegedy et al. (2016 Label Smoothing); Fedus et al. (2022 Router Z-Loss); Gemma / Grok (Logit Soft-Capping).

### Purpose & Topic Bridges
Deep architectures require careful regularizers and numerical stabilizers to prevent feature co-adaptation, reduce precision overflow (BF16/FP16), and stabilize gradient flow. This notebook implements Inverted Dropout, Stochastic Depth (DropPath), Logit Soft-Capping, Label Smoothing Cross-Entropy, and Mixture-of-Experts Router Z-Loss.

## 2. Mathematical Foundations & Governing Equations

1. **Inverted Dropout:** $y = \frac{x \odot m}{1 - p}$, preserving activation expectation $\mathbb{E}[y] = \mathbb{E}[x]$ during training and leaving inference untouched.
2. **DropPath (Stochastic Depth):** $x_{l+1} = x_l + \frac{b_l}{1 - p} F(x_l)$, dropping entire residual branches sample-wise to shorten effective depth during training.
3. **Logit Soft-Capping:** $y = C \cdot \tanh(x / C)$, smoothly clamping attention and readout logits within $(-C, C)$ to prevent BF16/FP16 overflow and softmax saturation.
4. **Label Smoothing:** $q_k = (1 - \alpha) \mathbb{I}(k=y) + \frac{\alpha}{K}$, penalizing overconfident peaky predictions by regularizing toward uniform entropy.
5. **Router Z-Loss:** $\mathcal{L}_z = \frac{c_z}{B} \sum_{b=1}^B (\log \sum_e e^{z_{b, e}})^2$, penalizing large router partition functions to stabilize MoE training.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/06_regularization_and_training_stabilizers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class RegularizationModule(nn.Module):
    def __init__(self, drop_p, droppath_p, softcap, z_coef):
        super().__init__()
        self.drop_p = float(drop_p[0])
        self.droppath_p = float(droppath_p[0])
        self.softcap = float(softcap[0])
        self.z_coef = float(z_coef[0])
        self.linear = nn.Linear(8, 4)

    def inverted_dropout(self, x, mask):
        return (x * mask) / (1.0 - self.drop_p)

    def droppath(self, x_id, x_br, mask):
        return x_id + (x_br * mask) / (1.0 - self.droppath_p)

    def softcapping(self, x):
        return self.softcap * torch.tanh(x / self.softcap)

    def label_smoothing_loss(self, logits, targets, alpha):
        return F.cross_entropy(logits, targets, label_smoothing=float(alpha))

    def router_z_loss(self, router_logits):
        return self.z_coef * torch.mean(torch.logsumexp(router_logits, dim=-1).pow(2))

    def forward(self, x): return self.linear(x)

model = RegularizationModule(weights['drop_rate'], weights['droppath_rate'], weights['softcap_val'], weights['z_coef']).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('RegularizationModule initialized successfully.')

In [ ]:
x_drop, drop_mask = weights['x_drop'], weights['drop_mask']
x_id, x_br, dp_mask = weights['x_identity'], weights['x_branch'], weights['droppath_mask']
x_log = weights['x_logits']
logits_ls, targets_ls = weights['logits_ls'], weights['targets_ls'].to(torch.int64)
alpha_ls = weights['smooth_alpha'][0]
router_logits = weights['router_logits']

y_dp = model.inverted_dropout(x_drop, drop_mask)
err_dp = (y_dp - weights['y_drop']).abs().max().item()
assert err_dp < 1e-5, f'Dropout parity error: {err_dp:.2e}'

y_dpath = model.droppath(x_id, x_br, dp_mask)
err_dpath = (y_dpath - weights['y_droppath']).abs().max().item()
assert err_dpath < 1e-5, f'DropPath parity error: {err_dpath:.2e}'

y_sc = model.softcapping(x_log)
err_sc = (y_sc - weights['y_softcap']).abs().max().item()
assert err_sc < 1e-5, f'SoftCap parity error: {err_sc:.2e}'

loss_ls = model.label_smoothing_loss(logits_ls, targets_ls, alpha_ls)
err_ls = (loss_ls - weights['loss_ls'][0]).abs().item()
assert err_ls < 1e-5, f'Label Smoothing parity error: {err_ls:.2e}'

loss_z = model.router_z_loss(router_logits)
err_z = (loss_z - weights['loss_z'][0]).abs().item()
assert err_z < 1e-5, f'Router Z-loss parity error: {err_z:.2e}'

y_anchor = model(test_in)
err_anchor = (y_anchor - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 6 invariants verified! Drop: {err_dp:.2e}, DropPath: {err_dpath:.2e}, SoftCap: {err_sc:.2e}, LS: {err_ls:.2e}, Z: {err_z:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Inverted Dropout': lambda: model.inverted_dropout(x_drop, drop_mask),
    'DropPath': lambda: model.droppath(x_id, x_br, dp_mask),
    'Logit Soft-Capping': lambda: model.softcapping(x_log),
    'Label Smoothing Loss': lambda: model.label_smoothing_loss(logits_ls, targets_ls, alpha_ls),
    'Router Z-Loss': lambda: model.router_z_loss(router_logits),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Pathological Failure Mode: Attention Logit Explosion & Gradient Vanishing vs Soft-Capping Gradient Health
z_raw = torch.tensor([[50.0, 0.0]], requires_grad=True, device=device)
prob_raw = F.softmax(z_raw, dim=-1)
loss_raw = prob_raw[0, 1] # Target is class 1
loss_raw.backward()
grad_raw = z_raw.grad[0, 0].abs().item()
print(f'[FAILURE DEMONSTRATED] Uncapped attention logit z=50 caused vanishing gradient: {grad_raw:.2e}')
assert grad_raw < 1e-15, f'Expected extreme gradient vanishing without capping, got {grad_raw:.2e}'

# With Logit Soft-Capping C=30.0, saturation is prevented and non-vanishing gradients propagate
z_capped = torch.tensor([[50.0, 0.0]], requires_grad=True, device=device)
prob_capped = F.softmax(model.softcapping(z_capped), dim=-1)
loss_capped = prob_capped[0, 1]
loss_capped.backward()
grad_capped = z_capped.grad[0, 0].abs().item()
print(f'[STABILIZED] Soft-capped logit maintains healthy gradient: {grad_capped:.4e}')
assert grad_capped > 1e-14, f'Expected healthy gradient with soft-capping, got {grad_capped:.4e}'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Stabilizers and linear anchor verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Numerical Stability in Reduced Precision:** Logit soft-capping prevents attention and readout values from exceeding BF16/FP16 limits, maintaining smooth softmax gradients.
2. **Effective Depth Control:** DropPath acts as a stochastic shortcut highway, enabling vision transformers with hundreds of layers to train with linear residual stability.
3. **Router Dispersion:** In Mixture-of-Experts, Router Z-loss constrains log-partition drift, preventing catastrophic round-off errors and token routing collapse.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 05_pooling_and_downsampling_operators](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/05_pooling_and_downsampling_operators.ipynb)